In [9]:
#Scraper for Untappd
import csv
import os,glob,sys, time
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def beer_data(i,beer):
    #print(beer,i)
    #attributes to be saved into CSV!
    beer_bid = beer.get_attribute("data-bid")
    beer_name = beer.find_element(By.CSS_SELECTOR, "p.name a").text
    beer_link = beer.find_element(By.CSS_SELECTOR, "p.name a").get_attribute("href")
    beer_abv = beer.find_element(By.CSS_SELECTOR, "p.abv").text
    beer_ibu = beer.find_element(By.CLASS_NAME, "ibu").text
    beer_rating = beer.find_element(By.CLASS_NAME, "rating-container").text
    beer_raters = beer.find_element(By.CLASS_NAME, "raters").text 
    beer_date = beer.find_element(By.CLASS_NAME, "date").text
    beer_brewery = beer.find_element(By.CLASS_NAME, "style").text
    beer_kind = beer.find_element(By.CSS_SELECTOR,"div.beer-details p.style:not(:has(a))").text
    beer_text = beer.find_element(By.XPATH, f".//p[contains(@class, 'desc-full') and contains(@class, '{beer_bid}')]").get_attribute("textContent")
    
    #print("\nbid:",beer_bid,"brewery",beer_brewery,"beer name",beer_name,"kind",beer_kind,"beer link:",beer_link,"abv:",beer_abv,"ibu:", beer_ibu,"rating:", beer_rating, "nr of raters:",beer_raters, "adding date:", beer_date)
    #data QA section, conversions, etc.
    #Elementary data
    if beer_bid and beer_name and beer_link and beer_brewery and beer_kind:
        print("")
    else:
        print("Failure in the main data of entry manual copy is required!")
        beer_bid = "N/A"
        beer_name = "N/A"
        beer_link = "N/A"
        beer_brewery = "N/A" 
        beer_kind = "N/A"
    #ABV
    if (beer_abv != "N/A") and (beer_abv):
        beer_abv = beer_abv.split("%")[0]
        beer_abv = float(beer_abv)
    else:
        beer_abv = "N/A"
    #IBU
    if (beer_ibu != "N/A IBU") and (beer_ibu):
        beer_ibu = beer_ibu.split(" ")[0]
        beer_ibu = int(beer_ibu)
    else:
        beer_ibu = "N/A"
    #Rating
    if beer_rating:
        beer_rating = beer_rating[1:-1]
        beer_rating = float(beer_rating)
    else:
        beer_rating = "N/A"
    #Raters
    if beer_raters:
        beer_raters = beer_raters.split(" ")[0]
        beer_raters = beer_raters.replace(",","")
        beer_raters = int(beer_raters)
    else:
        beer_raters = "N/A"
    #Adding date
    if beer_date:
        beer_date = beer_date.split(" ")[1]
    else:
        beer_date = "N/A"
    if beer_text:
        beer_text = beer_text.replace("\n", "")
        beer_text = beer_text.replace("Read Less", "")
    else:
        beer_text = "N/A"
    
    print("\nbid:",beer_bid,"brewery",beer_brewery,"beer name",beer_name,"kind",beer_kind,"beer link:",beer_link,"abv:",beer_abv,"ibu:", beer_ibu,"rating:", beer_rating, "nr of raters:",beer_raters, "adding date:", beer_date, "text",beer_text) 
    #step back to main page to iterate next beer
    ##driver.back()
    beer_drops = [
        i,
        beer_bid,
        beer_brewery,
        beer_name,
        beer_kind,
        beer_link,
        beer_abv,
        beer_ibu,
        beer_rating,
        beer_raters,
        beer_date,
        beer_text
    ]
    return beer_drops

def beer_to_csv(row):
    with open('Data.csv', 'a', newline='',encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(row)

#ChromeDriver folder
#CHROMEDRIVER_PATH = r"C:\Program Files\Google\Chrome\Application\chromedriver-win64\chromedriver.exe"

#URL
url = "https://untappd.com/beer/top_rated?country=hungary"
options = Options()
# options.add_argument("--headless=new") with headless it does not work somehow
driver = webdriver.Chrome()

try:
    #open url
    driver.get(url)

    #waiting 15s until the page loads
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
    print("Url loaded successfully", driver.title)
    
    wait = WebDriverWait(driver, 3)
    iframes = driver.find_elements(By.TAG_NAME, "iframe")
    print(f"Iframe-ek száma: {len(iframes)}")
    clicked = False

    for i, iframe in enumerate(iframes):
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame(iframe)

            elfogad = wait.until(
                EC.element_to_be_clickable((
                    By.XPATH,
                    "//button[normalize-space()='Elfogad' or @aria-label='Elfogad']"
                ))
            )

            elfogad.click()
            print(f"✅ Elfogad gomb kattintva (iframe #{i})")
            clicked = True
            break

        except (TimeoutException, NoSuchElementException):
            print(f"❌ Nincs Elfogad gomb az iframe #{i}-ben")

    if not clicked:
        print("⚠️ Nem találtam Elfogad gombot egyik iframe-ben sem. Futtasd újra!")
        sys.exit()
        
    #here starts the scraping
    driver.switch_to.default_content()           
    print("\nEnnyi sör van:")
    time.sleep(1)
    beers = driver.find_elements(By.CLASS_NAME,'beer-item')
    #beers = driver.find_elements(By.CSS_SELECTOR, ".beer-item")
    print(len(beers))

    #remove all the CSVs from the folder
    csv_files = glob.glob("*.csv")
    for file in csv_files:
        os.remove(file)
        print(f"Törölve: {file}")
    #create new CSV
    with open('Data.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['nr', 'bid','brewery','name','kind', 'link','abv','ibu', 'rating','nr_raters','adding_date','text'])
        
    for i in range (0,(len(beers))):
    #for i in range (0,2):
        time.sleep(2)
        beer=beers[i]
        beer_drops=beer_data(i,beer)
        beer_to_csv(beer_drops)
        
    #children = parent.find_elements(By.XPATH, "./div")
    #print(len(children))
    #time.sleep(15)

finally:
    #closing driver
    driver.quit()
    print("\nDriver Closed")

Url loaded successfully Top Rated Beers from Hungary – Top Rated Beers - Untappd
Iframe-ek száma: 8
❌ Nincs Elfogad gomb az iframe #0-ben
❌ Nincs Elfogad gomb az iframe #1-ben
❌ Nincs Elfogad gomb az iframe #2-ben
❌ Nincs Elfogad gomb az iframe #3-ben
❌ Nincs Elfogad gomb az iframe #4-ben
❌ Nincs Elfogad gomb az iframe #5-ben
❌ Nincs Elfogad gomb az iframe #6-ben
✅ Elfogad gomb kattintva (iframe #7)

Ennyi sör van:
50
Törölve: Data.csv


bid: 5826629 brewery B*BOP FERMENTORY beer name ÜÜÜÜÜÜÜÜ kind Sour - Smoothie / Pastry beer link: https://untappd.com/b/b-bop-fermentory-uuuuuuuu/5826629 abv: 4.5 ibu: N/A rating: 4.23 nr of raters: 944 adding date: 05/13/24 text Mango Popsicle Tart Smoothie Nu Sour.An ice cream inspired thick and juicy Smoothie Nu Sour Ale conditioned on a ton of Alphonso mangoes, ice cream cones, vanilla and a pinch of pakistani Halite salt.  


bid: 6030210 brewery HORIZONT Brewing beer name Night Shift Vintage 2024 - Imperial Pastry Stout Aged In Bourbon Barrels Wi